<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/dataset_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Precomputation and Loading for Titans

Этот ноутбук предназначен для фильтрации датасета TriviaQA, сохранения его в формате `ArrayRecord` и последующей быстрой загрузки для обучения модели Titans.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron

Cloning into 'kauldron'...
remote: Enumerating objects: 9668, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 9668 (delta 54), reused 41 (delta 32), pack-reused 9474 (from 3)
Receiving objects: 100% (9668/9668), 2.92 MiB | 15.79 MiB/s, done.
Resolving deltas: 100% (6934/6934), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 77.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 k

In [14]:
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog


Cloning into 'gemma'...
remote: Enumerating objects: 2564, done.
remote: Counting objects: 100% (1199/1199), done.
remote: Compressing objects: 100% (411/411), done.
remote: Total 2564 (delta 946), reused 819 (delta 788), pack-reused 1365 (from 2)
Receiving objects: 100% (2564/2564), 1.24 MiB | 26.92 MiB/s, done.
Resolving deltas: 100% (1800/1800), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.5/705.5 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [40]:
from gemma import gm
import jax
import jax.numpy as jnp

original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)


from gemma.gm.nn import _gemma

model = _gemma.Gemma3_1B(
    dtype=jnp.bfloat16,
    # return_last_only=False,
    tokens="batch.tokens",
)

In [18]:
import os
import grain.python as grain
from kauldron import kd
from etils import epath
import array_record.python.array_record_module as array_record
import tqdm
import pickle

# Константы
DATA_DIR = epath.Path('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')
MAX_LENGTH = 512 # длина последовательность в токенах
max_new_tokens = 64
BATCH_SIZE = 16 # Для сохранения не критично, но используется в оригинальном коде
MAX_CONTEXT_CHARS = (MAX_LENGTH - max_new_tokens) * 4
split_name = 'validation'

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Оригинальные компоненты и фильтрация

In [19]:
class FilterShortContext(grain.FilterTransform):
    def filter(self, x):
        ctxs = x.get('search_results', {}).get('search_context', [])
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_source_dataset(split_name):
    """Возвращает исходный TFDS датасет для фильтрации."""
    return kd.data.py.Tfds(
        name="trivia_qa/rc",
        split=split_name, ##splits: 'test' | 'train' | 'validation'
        shuffle=False, # Для сохранения порядок не важен
        num_epochs=1,  # Проходим один раз
        batch_size=None, # Читаем по одной записи
    )

## 2. Сохранение в ArrayRecord

Этот этап нужно запустить один раз.

In [12]:
split_name = 'test'
def precompute_and_save():
    ds = get_source_dataset(split_name)
    filter_transform = FilterShortContext()

    output_path = DATA_DIR / f"{split_name}.array_record"
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    print(f"Начинаю фильтрацию и сохранение в {output_path}...")

    count = 0
    # Итерируемся по сырым данным из TFDS
    for record in tqdm.tqdm(ds):
        if filter_transform.filter(record):
            # Сохраняем только нужные поля для экономии места
            serialized = pickle.dumps(record)
            writer.write(serialized)
            count += 1

    writer.close()
    print(f"Готово! Сохранено {count} валидных записей.")

precompute_and_save() # Раскомментируйте для запуска

Начинаю фильтрацию и сохранение в /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test.array_record...


  1%|          | 142/17210 [00:00<00:12, 1403.44it/s]

Disabling pygrain multi-processing (unsupported in colab).


100%|██████████| 17210/17210 [00:12<00:00, 1372.10it/s]

Готово! Сохранено 6143 валидных записей.


## 3. Загрузка сохраненных данных

Эта функция заменяет ваш `get_train_dataset_tydi_qa` в основном цикле обучения.

In [13]:
def get_precomputed_dataset(batch_size=16, tokenizer=None,  files=None):
    """Загружает и объединяет отфильтрованные данные из нескольких файлов.

    Args:
        batch_size: Размер батча.
        tokenizer: Токенизатор Gemma.
        files: Список имен файлов, например ['train.array_record', 'validation.array_record']
    """
    if files is None:
        files = [f'{split_name}.array_record']

    paths = [str(DATA_DIR / f) for f in files]
    print(f"Загрузка данных из: {paths}")

    # 1. Определяем источник данных (DataSource)
    class PickledArrayRecordDataSource(grain.python.ArrayRecordDataSource):
        def __getitem__(self, idx):
            return pickle.loads(super().__getitem__(idx))

    data_source = PickledArrayRecordDataSource(paths)

    # 2. Создаем Kauldron Dataset
    return kd.data.py.DataSource(
        data_source=data_source,
        shuffle=True,      # Перемешивание важно при объединении разных сплитов!
        num_epochs=None,   # Бесконечно для обучения
        batch_size=batch_size,
        transforms=[
            # Здесь применяются format_triviaqa и gm.data.Seq2SeqTask
            kd.data.py.Elements(keep=["tokens", "target", "loss_mask"]),
        ],
    )

## 4. Использование в kd.train.Trainer

Пример того, как это интегрируется в основной конфиг.

In [ ]:
'''
trainer = kd.train.Trainer(
    seed=42,
    workdir='./workdir',
    train_ds=get_precomputed_dataset(
        batch_size=BATCH_SIZE,
        tokenizer=tokenizer,
        max_length=MAX_CONTEXT_CHARS,
        files=['train.array_record', 'validation.array_record', 'test.array_record']
    ),
    model=model,
    # ... остальная конфигурация
)
'''

## 5. Офлайн-генерация ответов Gemma

В этом разделе мы пропускаем отфильтрованные записи `trivia_qa/rc` через оригинальную модель `Gemma3_1B_IT` для генерации её собственных ответов (`dst`).

In [31]:
input_file = f"{split_name}.array_record"
output_file = f"{split_name}_gemma_generated.array_record"
input_path = os.path.join(str(DATA_DIR), input_file)
reader = array_record.ArrayRecordReader(str(input_path))

In [35]:
reader.num_records()

6707

In [38]:
# ВАЖНО: Эту ячейку нужно запускать в окружении с загруженной моделью Gemma.

import array_record.python.array_record_module as array_record
import pickle
import tqdm
import kauldron as kd
from gemma import gm
from etils import epath
import os

# 1. Функция форматирования промпта
def format_triviaqa_prompt(x):
    ctx = ""
    if x.get('search_results', {}).get('search_context', []):
        ctx = x['search_results']['search_context'][0]
    q = x["question"]
    return f"Context: {ctx}\n\nQuestion: {q}\n\nAnswer: "

# 2. Функция генерации датасета
def generate_and_save_offline_dataset(
    model,
    params,
    tokenizer,
    split_name,
    max_new_tokens=max_new_tokens
):
    input_file = f"{split_name}.array_record"
    output_file = f"{split_name}_gemma_generated.array_record"
    input_path = os.path.join(str(DATA_DIR), input_file)
    output_path = os.path.join(str(DATA_DIR), output_file)

    print(f"Чтение из {input_path}")
    print(f"Сохранение в {output_path}")

    # Инициализация семплера Gemma
    sampler = gm.text.Sampler(
        model=model,
        params=params,
        tokenizer=tokenizer,
    )

    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    # Чтение исходного ArrayRecord
    reader = array_record.ArrayRecordReader(str(input_path))

    print(f"Всего записей: {reader.num_records()}")

    # ArrayRecordReader не итерируемый напрямую, используем range
    for i in tqdm.tqdm(range(reader.num_records())):
        record_bytes = reader.read()
        record = pickle.loads(record_bytes)

        # Формируем prompt
        prompt = format_triviaqa_prompt(record)

        # Генерируем ответ (offline)
        response_text = sampler.sample(prompt, max_new_tokens=max_new_tokens)

        # Подменяем эталонный ответ сгенерированным
        if 'answer' not in record:
            record['answer'] = {}
        record['answer']['value'] = response_text

        # Записываем обратно
        writer.write(pickle.dumps(record))

    writer.close()
    print(f"Успешно сгенерировано и сохранено {len(reader)} записей.")

In [ ]:
generate_and_save_offline_dataset(
    model=model,
    params=original_params,
    tokenizer= gm.text.Gemma3Tokenizer(),
    split_name='test',
    max_new_tokens=max_new_tokens

)

Чтение из /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test.array_record
Сохранение в /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test_gemma_generated.array_record
Всего записей: 6143


  0%|          | 13/6143 [02:44<14:37:18,  8.59s/it]